# Local search and optimisation

The algorithms of [search](search.ipynb) return a **path**: the sequence of actions from the start to the goal. In many problems the path doesn't matter, only the final state: a timetable, the layout of a circuit, the positions of queens on a board. **Local search** starts from a complete candidate, however bad, and improves it step by step, guided by an **objective function** that scores each candidate.

It keeps only the current candidate in memory, never a tree of paths, so it works in state spaces far too large to search systematically. The price: it can get stuck, and it can't prove that no better solution exists.

## The eight-queens problem

Place eight queens on a chessboard so that no two attack each other: no two in the same row, column or diagonal. With one queen per column, a candidate is a list of eight row numbers, one per column, and there are 8⁸ = 16,777,216 candidates. The objective to minimise is the number of pairs of queens that attack each other; a solution has 0.

In [ ]:
SIZE = 8


def attacking_pairs(rows):
    pairs = 0
    for column_a in range(SIZE):
        for column_b in range(column_a + 1, SIZE):
            same_row = rows[column_a] == rows[column_b]
            same_diagonal = abs(rows[column_a] - rows[column_b]) == column_b - column_a
            pairs += same_row or same_diagonal
    return pairs


print(attacking_pairs([0, 1, 2, 3, 4, 5, 6, 7]), "attacking pairs with all the queens on one diagonal")

28 attacking pairs with all the queens on one diagonal


The **neighbours** of a candidate are the candidates that differ in one move: one queen moved to another row of its column. Each candidate has 8 × 7 = 56 neighbours.

In [ ]:
def neighbours(rows):
    for column in range(SIZE):
        for row in range(SIZE):
            if row != rows[column]:
                yield rows[:column] + [row] + rows[column + 1:]

## Hill climbing

**Hill climbing**, here as *steepest descent* because the objective is a cost, moves to the best neighbour as long as it is better than the current candidate, and stops when none is. Where it stops is a **local minimum**: no single move improves it, but it may not be a solution.

In [ ]:
def hill_climb(rows):
    steps = 0
    while True:
        best = min(neighbours(rows), key=attacking_pairs)
        if attacking_pairs(best) >= attacking_pairs(rows):
            return rows, steps
        rows, steps = best, steps + 1

A thousand random starting boards, with a fixed seed so that the numbers are the same on every run:

In [ ]:
import random

random.seed(0)
starts = [[random.randrange(SIZE) for _ in range(SIZE)] for _ in range(1000)]
results = [hill_climb(start) for start in starts]
solved = sum(attacking_pairs(rows) == 0 for rows, _ in results)
average_steps = sum(steps for _, steps in results) / len(results)
print(f"Solved {solved} of {len(starts)} boards, in {average_steps:.1f} steps on average")

Solved 161 of 1000 boards, in 3.3 steps on average


Hill climbing is fast, a few steps per board, and it fails most of the time: most starting boards lead to a local minimum with a pair of queens still attacking each other. Russell and Norvig's textbook reports the same behaviour, about 14% of boards solved, with a slightly different rule for ties.

## Random restarts

The simplest remedy: when hill climbing gets stuck, start again from a new random board, until one run reaches a solution.

In [ ]:
random.seed(1)
restarts = 0
while True:
    restarts += 1
    rows, _ = hill_climb([random.randrange(SIZE) for _ in range(SIZE)])
    if attacking_pairs(rows) == 0:
        break
print(f"Solved after {restarts} starts: {rows}")

Solved after 4 starts: [5, 1, 6, 0, 2, 4, 7, 3]


## Simulated annealing

**Simulated annealing** (Kirkpatrick, Gelatt and Vecchi, 1983) lets the search escape local minima by sometimes accepting a **worse** move. It picks a random neighbour; a better one is always accepted, and a worse one with probability e^(−Δ/T), where Δ is how much worse it is and *T* a **temperature** that falls step by step. At first the search wanders almost freely; as it cools, it behaves more and more like hill climbing. The name comes from metallurgy, where slowly cooled metal settles into a low-energy, regular structure.

In [ ]:
import math


def simulated_annealing(rows, temperature=2.0, cooling=0.995, max_steps=20_000):
    cost = attacking_pairs(rows)
    for step in range(max_steps):
        if cost == 0:
            return rows, step
        column, row = random.randrange(SIZE), random.randrange(SIZE)
        candidate = rows[:column] + [row] + rows[column + 1:]
        change = attacking_pairs(candidate) - cost
        if change <= 0 or random.random() < math.exp(-change / temperature):
            rows, cost = candidate, cost + change
        temperature = max(temperature * cooling, 1e-3)
    return rows, max_steps

In [ ]:
random.seed(0)
annealing_results = [simulated_annealing(start) for start in starts[:200]]
solved = sum(attacking_pairs(rows) == 0 for rows, _ in annealing_results)
print(f"Solved {solved} of 200 boards")

Solved 192 of 200 boards


On the same starting boards, accepting some worse moves turns most of the failures into solutions, at the cost of many more, cheaper steps: each looks at one random neighbour instead of all 56.

## Genetic algorithms

A **genetic algorithm** (John Holland, 1975) keeps a whole **population** of candidates and imitates natural selection. In each generation, the fittest half survive as parents; each child takes the first part of one parent and the rest of another (**crossover**), and sometimes a random change (**mutation**). Crossover can combine good parts found by different candidates, which a single local search can't.

In [ ]:
def genetic_algorithm(population_size=100, mutation_rate=0.2, max_generations=1000):
    population = [[random.randrange(SIZE) for _ in range(SIZE)] for _ in range(population_size)]
    for generation in range(max_generations):
        population.sort(key=attacking_pairs)
        if attacking_pairs(population[0]) == 0:
            return population[0], generation
        parents = population[: population_size // 2]
        children = []
        while len(children) < population_size:
            mother, father = random.sample(parents, 2)
            cut = random.randrange(1, SIZE)
            child = mother[:cut] + father[cut:]
            if random.random() < mutation_rate:
                child[random.randrange(SIZE)] = random.randrange(SIZE)
            children.append(child)
        population = children
    return population[0], max_generations

In [ ]:
random.seed(0)
runs = [genetic_algorithm() for _ in range(20)]
generations = sorted(generation for _, generation in runs)
print("Solved", sum(attacking_pairs(best) == 0 for best, _ in runs), "of 20 runs")
print("Generations needed:", generations)

Solved 19 of 20 runs
Generations needed: [6, 9, 10, 12, 12, 13, 13, 13, 14, 14, 16, 23, 33, 34, 35, 38, 43, 451, 471, 1000]


Most runs find a solution in a few dozen generations; a few wander for hundreds, and one reached the limit of 1,000 without finding one. Genetic algorithms need tuning, population size, mutation rate, how parents are chosen, and no single setting works best for every problem.

## From local search to machine learning

Training a machine-learning model is local search too, on a continuous objective. The candidate is the model's parameters, the objective is the error on the training data, and **gradient descent** moves to a better neighbour in the direction that the gradient says lowers the error fastest ([how a neural network learns](../../background/neural_networks.md#how-it-learns)). The same questions arise: how big a step, and how to avoid getting stuck. Simulated annealing's idea of noise that helps escape bad regions has a cousin in the randomness of stochastic gradient descent.